In [1]:
import pandas as pd
from graph import Graph
from flight import Flight
from airport import Airport
from pnr import PNR
from datetime import datetime
import numpy as np
from dimod import BinaryQuadraticModel
import dimod
import itertools
import singularity.optimization as sop


SINGULARITY_TOKEN = "c8b07936-99b0-416f-a749-150da42f6aa1"
DWAVE_API_TOKEN = "DEV-7b444792d7a75ce33cc91f50fac2e0e021db9259"

C:\Users\eduar\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## We load the flight and passenger information

In [2]:
import pickle
with open("pnr_list.pkl", "rb") as f:
    pnr_list = pickle.load(f)
    
with open("available_flights.pkl", "rb") as f:
    available_flights = pickle.load(f)

with open("flights_out_in_dic.pkl", "rb") as f:
    flights_out_in_dic = pickle.load(f)
    
with open("flights_2legs_out_in_dic.pkl", "rb") as f:
    flights_2legs_out_in_dic = pickle.load(f)

In [3]:
## We now define the time penalty computation, 
#### The leg_penalty variable is for us to add to multi-legt flights
#### Those will have a high leg_penalty, as we prefer direct flights
def time_penalty(time_difference: float) -> float:
        if time_difference < 0:
            return -10000
        elif time_difference <= 360:
            #return (70 - time_difference)
            return 70
        elif time_difference <= 720:
            #return (50 - time_difference)
            return 50
        elif time_difference <= 1440:
            #return (40 - time_difference)
            return 40
        elif time_difference <= 2880:
            #return (30 - time_difference)
            return 30
        else:
            return -10000

In [4]:
## We now define the local constraint function
#### This function adds the constraint that only one flight 
#### Is given to a given passenger
# variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_one"].append( ( passenger.trip_id, flight.dep_key ) )
def local_constraint(variables_per_passenger, name, penalty_strength = 100):
    
    reaccommodations_passenger = variables_per_passenger["one_one"] + variables_per_passenger["forced_one"] + variables_per_passenger["multi_one"]
    reaccommodations_passenger = reaccommodations_passenger + variables_per_passenger["one_multi"] + variables_per_passenger["multi_multi"]
    name = name
    # We only want for the passenger to be assigned ONCE!!! Therefore the sum must be one

    ancilla_variable = sop.Variable("ancilla")
    local_cost = 0*ancilla_variable
    #print((type(local_cost)))
    #print(Total_cost)

    for variable in reaccommodations_passenger:
        #print("SIUUU")
        #print(type(variable))
        local_cost = local_cost + variable

    return sop.Constraint(lhs = local_cost, operator = "==", rhs = 1, 
                          penalty_strength = penalty_strength, 
                          name = name )

In [5]:
def seat_constraint(variables_per_flight, available_flights, penalty_strength = 1000):
    # List to save the constraints
    constraints = []

    # We now iterate over all the available flights
    for flight in available_flights:
        title = r"Flight_{}_Seat_Constraint".format(flight.dep_key)
        avail_seats = flight.c_avail_cnt + flight.y_avail_cnt

        # Seats taken by the variables for the assigned flight
        ancilla_variable = sop.Variable("ancilla")
        # Now Total_cost is in terms of SOP!!!
        taken_seats = 0*ancilla_variable

        for double in variables_per_flight[flight.dep_key]:
            if double == []:
                continue
            else:
                # Here we have the variable
                variable = double[0]
                # Here we have the passenger
                passenger = double[1]
                taken_seats = taken_seats + passenger.pax_cnt * variable


            # Now we do the constraints
        constraint = sop.Constraint(lhs = taken_seats, operator="<=", 
                           rhs = avail_seats, penalty_strength= penalty_strength,
                           name = title)
        constraints.append(constraint)

    return constraints

constraints = []
penalty_strength = 100
    # We now iterate over all the available flights
for flight in available_flights:
    print("Flight: ", flight.dep_key)
    title = r"Flight_{}_Seat_Constraint".format(flight.dep_key)
    avail_seats = flight.c_avail_cnt + flight.y_avail_cnt

    # Seats taken by the variables for the assigned flight
    ancilla_variable = sop.Variable("ancilla")
    # Now Total_cost is in terms of SOP!!!
    taken_seats = 0*ancilla_variable

    for double in variables_per_flight[flight.dep_key]:
        
        if double == []:
            print("Skipping!")
            continue
        else:
            # Here we have the variable
            variable = double[0]
            print("Variable: ", variable)
            # Here we have the passenger
            passenger = double[1]
            taken_seats = taken_seats + passenger.pax_cnt * variable
            print("Taken seats: ", taken_seats)


            # Now we do the constraints
            constraint = sop.Constraint(lhs = taken_seats, operator="<=", 
                           rhs = avail_seats, penalty_strength= penalty_strength,
                           name = title)
            constraints.append(constraint)

In [6]:
def seat_constraint(variables_per_flight, available_flights, penalty_strength = 1000):
    constraints = []
    penalty_strength = 100
        # We now iterate over all the available flights
    for flight in available_flights:
        #print("Flight: ", flight.dep_key)
        title = r"Flight_{}_Seat_Constraint".format(flight.dep_key)
        avail_seats = flight.c_avail_cnt + flight.y_avail_cnt

        # Seats taken by the variables for the assigned flight
        ancilla_variable = sop.Variable("ancilla")
        # Now Total_cost is in terms of SOP!!!
        taken_seats = 0*ancilla_variable

        for double in variables_per_flight[flight.dep_key]:

            if double == []:
                print("Skipping!")
                continue
            else:
                # Here we have the variable
                variable = double[0]
                #print("Variable: ", variable)
                # Here we have the passenger
                passenger = double[1]
                taken_seats = taken_seats + passenger.pax_cnt * variable
                #print("Taken seats: ", taken_seats)


                # Now we do the constraints
                constraint = sop.Constraint(lhs = taken_seats, operator="<=", 
                               rhs = avail_seats, penalty_strength= penalty_strength,
                               name = title)
                constraints.append(constraint)
    return constraints

In [7]:
## We create a second version of the seat_constraints 
## Which directly ADDS the seat constraints rather than just creating them

def add_seat_constraint(variables_per_flight, available_flights, model, penalty_strength = 1000):
    for flight in available_flights:
        title = r"Flight_{}_Seat_Constraint".format(flight.dep_key)
        avail_seats = flight.c_avail_cnt + flight.y_avail_cnt

        # Seats taken by the variables for the assigned flight
        ancilla_variable = sop.Variable("ancilla")
        # Now Total_cost is in terms of SOP!!!
        taken_seats = 0*ancilla_variable

        for double in variables_per_flight[flight.dep_key]:

            # Here we have the variable
            variable = double[0]
            # Here we have the passenger
            passenger = double[1]
            taken_seats = taken_seats + passenger.pax_cnt * variable


        # Now we do the constraints
        constraint = sop.Constraint(lhs = taken_seats, operator="<=", 
                       rhs = avail_seats, penalty_strength= penalty_strength,
                       name = title)
        
        # Now we add the constraint to the model
        model.add_constraint(constraint)

    # We return the model!
    return model

In [8]:
# We add costs for the CVM variables

one_one_cost = 25
one_multi_cost = int(15)
multi_one_cost = int(20)
multi_multi_cost = int(10)

# Factor to include in the computation of the cvm!
cvm_factor = {}
cvm_factor["one_one"] = int(25)
cvm_factor["one_multi"] = int(15)
cvm_factor["multi_one"] = int(20)
cvm_factor["multi_multi"] = int(10)
cvm_factor["forced_one"] = int(30)

# To change this cost, depending on the feedback that we obtain
forced_one_cost = 30
## Different penalties for each proposed solution

In [9]:
n_variables = 0
n_one_one = 0
n_one_multi = 0
n_multi_multi = 0
n_multi_one = 0
n_forced_one = 0

In [10]:
print(" Setting seach space! ")

pnr_list = pnr_list[0:2]
#available_flights = available_flights[0:10]

print("Reduced dataset: ")
print("Number of passengers: ", len(pnr_list) )
print("Number of available flights: ", len(available_flights) )

 Setting seach space! 
Reduced dataset: 
Number of passengers:  2
Number of available flights:  3221


In [11]:
## Now we create the dictionaries! 
# For local constraints
variables_per_passenger = {}
# For seat constraints!
variables_per_flight = {}

# We initialize / initialise the variables_per_flight dictionary
for flight in available_flights:
    variables_per_flight[str(flight.dep_key)] = []



# We create the general dictionary 
general_dic = {}
general_dic["one_one"] = []
general_dic["one_multi"] = []
general_dic["multi_one"] = []
general_dic["multi_multi"] = []
general_dic["forced_one"] = []


In [12]:
constraints_list = []

In [13]:
# We have the TOTAL cost for the whole system
ancilla_variable = sop.Variable("ancilla")

# Now Total_cost is in terms of SOP!!!
Total_cost = 0*ancilla_variable
#print((type(Total_cost)))
#print(Total_cost)

variables_list = []

constraints_pnr = {}

In [14]:
pnr_list[0].booked_multi_leg

False

In [15]:
flights_out_in_dic[("EBG", "TPH")]

[Flight 5654 (EBG -> TPH):
   Departure Date: 2027-10-26
   Departure Time (Local): 2027-10-26 12:44, (Zulu): 2027-10-26 16:44
   Arrival Time (Local): 2027-10-26 14:13, (Zulu): 2027-10-26 16:44
   Cabin C: Capacity: 16, Occupied: 16, Passengers: 0, Available: 16
   Cabin Y: Capacity: 150, Occupied: 150, Passengers: 112, Available: 38
   Departure Key: AZ20271026EBGTPH5654,
 Flight 5654 (EBG -> TPH):
   Departure Date: 2027-10-22
   Departure Time (Local): 2027-10-22 12:44, (Zulu): 2027-10-22 16:44
   Arrival Time (Local): 2027-10-22 14:13, (Zulu): 2027-10-22 16:44
   Cabin C: Capacity: 16, Occupied: 16, Passengers: 12, Available: 4
   Cabin Y: Capacity: 150, Occupied: 150, Passengers: 148, Available: 2
   Departure Key: AZ20271022EBGTPH5654,
 Flight 5654 (EBG -> TPH):
   Departure Date: 2027-10-24
   Departure Time (Local): 2027-10-24 12:44, (Zulu): 2027-10-24 16:44
   Arrival Time (Local): 2027-10-24 14:13, (Zulu): 2027-10-24 16:44
   Cabin C: Capacity: 16, Occupied: 16, Passengers: 

In [16]:
pnr_list[1]

PNR(AZ20271021TPHRHP12465, 2027-10-21,Time 2027-10-21 23:25 -> 2027-10-22 2:55 ,Y, 21, TPH -> RHP)

In [17]:
for passenger in pnr_list: 
    print("(Passenger, trip number): ", (passenger.recloc, passenger.trip_number[0]), "/", len(pnr_list))

    variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))] = general_dic
    
    #***********************************************************************************
    ### DIRECT FLIGHTS
    #***********************************************************************************

    ## ONE-ONE CASE
    ## BOOKING ONLY HAS ONE DIRECT FLIGHT
    ## THEREFORE, WE JUST FIND A NEW DIRECT FLIGHT
    if not passenger.booked_multi_leg:

        # We retrieve the flights important for our case
        available_flights_case = flights_out_in_dic[ (str(passenger.orig_cd), str(passenger.dest_cd)) ]
        
        # Now we iterate over all possible case
        for flight in available_flights_case:

            # Retrieve the new time departures
            new_time_dep = datetime.strptime(flight.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            new_time_arr = datetime.strptime(flight.arr_dtmz, "%Y-%m-%d %H:%M") 
            
            # Retrieve the original time departures
            orig_time_dep = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")
            orig_time_arr = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")
                
            time_diff_dep = ((new_time_dep - orig_time_dep).total_seconds() / 60.0) # Minutes
            time_diff_arr = ((new_time_arr - orig_time_arr).total_seconds() / 60.0) 

            cost_dep = time_penalty(time_diff_dep)
            cost_arr = time_penalty(time_diff_arr)

            # Is this a good pairing?
            if ((cost_dep <= 0) or (cost_arr <= 0)): # Check the penalty
                #  We do not add an energy for this flight, the flight is already terrible!
                continue 
            else:
                # We add a variable for this flight
                var = (passenger.trip_id, flight.dep_key)
                 
                # We put it in shape of the sop language
                var = sop.Variable(str(var))
                #print(var)
                var_cost = cost_dep + cost_arr + cvm_factor["one_one"]*passenger.cvm
                
                # We save the variable
                variables_list.append((var, var_cost))

                # Now we add it to the total cost
                Total_cost = Total_cost + var * var_cost

                # We save the assigned DIRECT flight for this passenger
                variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_one"].append( var )

                # We add the assigned flight to be dictionary saving the global variables
                variables_per_flight[flight.dep_key].append( (var, passenger) )

                # We increment the number of total variables
                n_variables = n_variables + 1
                n_one_one = n_one_one + 1

    
    if passenger.booked_multi_leg:
        ## FORCED-ONE CASE
        ## BOOKING HAS SEVERAL LEGS, AND WE OBTAIN A DIRECT ONE 
        ## FOR THE WHOLE TRIP, NOT JUST THE CANCELLED LEG    
        
        # We retrieve the flights important for our case
        available_flights_case = flights_out_in_dic[ (str(passenger.oper_od_orig_cd), str(passenger.oper_od_dest_cd)) ]
        for flight in available_flights_case:

            # Retrieve the new time departures
            new_time_dep = datetime.strptime(flight.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            new_time_arr = datetime.strptime(flight.arr_dtmz, "%Y-%m-%d %H:%M") 
            
            # Passenger originally BOOKED multiple legs!
            ideal_time_dep = datetime.strptime(passenger.ideal_dep_dtmz, "%Y-%m-%d %H:%M")
            ideal_time_arr = datetime.strptime(passenger.ideal_arr_dtmz, "%Y-%m-%d %H:%M")

            ideal_time_diff_dep = ((new_time_dep - ideal_time_dep).total_seconds() / 60.0) # Mintues
            ideal_time_diff_arr = ((new_time_arr - ideal_time_arr).total_seconds() / 60.0)

            ideal_cost_dep = time_penalty(ideal_time_diff_dep)
            ideal_cost_arr = time_penalty(ideal_time_diff_arr)
                
            # Is this flight terrible?
            if ((ideal_cost_dep <= 0) or (ideal_cost_arr <= 0)): # Check the penalty
                #  We do not add an energy for this flight, the flight is already terrible!
                continue 
            else:
                # We add a variable for this flight
                var = (passenger.trip_id, flight.dep_key)
                 
                # We put it in shape of the sop language
                var = sop.Variable(str(var))
                #print(var)
                var_cost = cost_dep + cost_arr + cvm_factor["forced_one"]*passenger.cvm
                
                # We save the variable
                variables_list.append((var, var_cost))

                # Now we add it to the total cost
                Total_cost = Total_cost + var * var_cost

                # We save the assigned DIRECT flight for this passenger
                variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["forced_one"].append( var )

                # We add the assigned flight to be dictionary saving the global variables
                variables_per_flight[flight.dep_key].append( (var, passenger) )

                # We increment the number of total variables
                n_variables = n_variables + 1
                n_forced_one = n_forced_one + 1
     

        #MULTI-ONE CASE
        ## BOOKING HAS SEVERAL LEGS, AND WE OBTAIN A DIRECT ONE
        ## FOR THE CANCELLED TRIP
        available_flights_case = flights_out_in_dic[ (str(passenger.orig_cd), str(passenger.dest_cd)) ]
        for flight in available_flights_case:

            # Retrieve the new departure and arrival times
            new_time_dep = datetime.strptime(flight.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            new_time_arr = datetime.strptime(flight.arr_dtmz, "%Y-%m-%d %H:%M") 

            # Compute the new departure and arrival times
            orig_time_dep = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")
            orig_time_arr = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")
            
            time_diff_dep = ((new_time_dep - orig_time_dep).total_seconds() / 60.0) # Minutes
            time_diff_arr = ((new_time_arr - orig_time_arr).total_seconds() / 60.0) 

            cost_dep = time_penalty(time_diff_dep)
            cost_arr = time_penalty(time_diff_arr)

            # Is this a good pairing?
            if ((cost_dep <= 0) or (cost_arr <= 0)): # Check the penalty
                    #  We do not add an energy for this flight, the flight is already terrible!
                continue 
            else:

                # We add a variable for this flight
                var = (passenger.trip_id, flight.dep_key)
                 
                # We put it in shape of the sop language
                var = sop.Variable(str(var))
                #print(var)
                var_cost = cost_dep + cost_arr + cvm_factor["multi_one"]*passenger.cvm
                
                # We save the variable
                variables_list.append((var, var_cost))

                # Now we add it to the total cost
                Total_cost = Total_cost + var * var_cost

                # We save the assigned DIRECT flight for this passenger
                variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["multi_one"].append( var )

                # We add the assigned flight to be dictionary saving the global variables
                variables_per_flight[flight.dep_key].append( (var, passenger) )
                
                # We increment the number of total variables
                n_variables = n_variables + 1
                n_multi_one = n_multi_one + 1
    

    #****************************************************************************************************
    # Now we do NON-DIRECT flights
    #****************************************************************************************************

    ##############################################
    ## ONE-MULTI CASE
    ###############################################
    if not passenger.booked_multi_leg:
        
        available_flights_case = flights_2legs_out_in_dic[(str(passenger.oper_od_orig_cd), str(passenger.oper_od_dest_cd))]
        # Now we iterate over the flights
        for flight_combination in available_flights_case:
            
            first_leg = flight_combination[0]
            second_leg = flight_combination[1]
            
            # Check the departure penalty costs FIRST before doiny any computation
            original_time_dep = datetime.strptime(passenger.ideal_dep_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_dep = datetime.strptime(first_leg.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            time_diff_dep = ((new_time_dep - original_time_dep).total_seconds() / 60.0)  # in minutes
                        
            # Check if the penalty is acceptable or not
            cost_dep = time_penalty(time_diff_dep)
            if (cost_dep <= 0): # Check the penalty
                # Skip all the pairings with ths flight as the first leg
                continue

            # Now we check if the second flight is trash in terms of arriving
            original_time_arr = datetime.strptime(passenger.ideal_arr_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_arr = datetime.strptime(second_leg.arr_dtmz, "%Y-%m-%d %H:%M")
            time_diff_arr = ((new_time_arr - original_time_arr).total_seconds() / 60.0)  # in minutes

            # Check if the cost is acceptable or not
            cost_arr = time_penalty(time_diff_arr)
            if (cost_arr <= 0):
                # Skip this second leg
                continue
            else:
                # Now we add the variable!
                var = (passenger.trip_id, (first_leg.dep_key, second_leg.dep_key))
                # We put it in shape of the sop language
                var = sop.Variable(str(var))
                #print(var)
                var_cost = cost_dep + cost_arr + cvm_factor["one_multi"]*passenger.cvm
                    
                # We save the variable
                variables_list.append((var, var_cost))

                # Now we add it to the total cost
                Total_cost = Total_cost + var * var_cost

                # We save the assigned DIRECT flight for this passenger
                variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["one_multi"].append( var )

                # We add the assigned flight to be dictionary saving the global variables
                variables_per_flight[first_leg.dep_key].append( (var, passenger) )
                variables_per_flight[second_leg.dep_key].append( (var, passenger) )

                # We increment the number of total variables
                n_variables = n_variables + 1
                n_one_multi = n_one_multi + 1


    if passenger.booked_multi_leg:
        #****************************************************************************************************************
            ## MULTI-MULTI COST!!! ##
        #****************************************************************************************************************
        available_flights_case = flights_2legs_out_in_dic[(str(passenger.orig_cd), str(passenger.dest_cd))]
        # Now we iterate over the flights
        for flight_combination in available_flights_case:
            
            first_leg = flight_combination[0]
            second_leg = flight_combination[1]
            
            # Check the departure penalty costs FIRST before doiny any computation
            original_time_dep = datetime.strptime(passenger.dep_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_dep = datetime.strptime(first_leg.dep_dtmz, "%Y-%m-%d %H:%M")  # flight time does include seconds
            time_diff_dep = ((new_time_dep - original_time_dep).total_seconds() / 60.0)  # in minutes
                        
            # Check if the penalty is acceptable or not
            cost_dep = time_penalty(time_diff_dep)
            if (cost_dep <= 0): # Check the penalty
                # Skip all the pairings with ths flight as the first leg
                continue

            # Now we check if the second flight is trash in terms of arriving
            original_time_arr = datetime.strptime(passenger.arr_dtmz, "%Y-%m-%d %H:%M")  # pnr does not include seconds
            new_time_arr = datetime.strptime(second_leg.arr_dtmz, "%Y-%m-%d %H:%M")
            time_diff_arr = ((new_time_arr - original_time_arr).total_seconds() / 60.0)  # in minutes

            # Check if the cost is acceptable or not
            cost_arr = time_penalty(time_diff_arr)
            if (cost_arr <= 0):
                # Skip this second leg
                continue
            else:
                # Now we add the variable!
                var = (passenger.trip_id, (first_leg.dep_key, second_leg.dep_key))
                # We put it in shape of the sop language
                var = sop.Variable(str(var))
                #print(var)
                var_cost = cost_dep + cost_arr + cvm_factor["multi_multi"]*passenger.cvm
                    
                # We save the variable
                variables_list.append((var, var_cost))

                # Now we add it to the total cost
                Total_cost = Total_cost + var * var_cost

                # We save the assigned DIRECT flight for this passenger
                variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))]["multi_multi"].append( var )

                # We add the assigned flight to be dictionary saving the global variables
                variables_per_flight[first_leg.dep_key].append( (var, passenger) )
                variables_per_flight[second_leg.dep_key].append( (var, passenger) )

                # We increment the number of total variables
                n_variables = n_variables + 1
                n_one_multi = n_one_multi + 1
        
            
    
    # We now SAVE the local constraints
    #print( variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))] )
    #print("Variables per passenger: ", variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))])
    print("Adding local constraints!")
    constraints_pnr[str(passenger.recloc)] = local_constraint(variables_per_passenger[(str(passenger.recloc), str(passenger.trip_number[0]))], name = passenger.recloc)
    constraints_list.append( constraints_pnr[str(passenger.recloc)] )

(Passenger, trip number):  (0, 1) / 2
Adding local constraints!
(Passenger, trip number):  (1, 1) / 2
Adding local constraints!


In [18]:
# Now we obtain the seat constraints
seat_constraints = seat_constraint(variables_per_flight, available_flights, penalty_strength = 1000)

In [19]:
# Vector saving the total_constraints
total_constraints = constraints_list + seat_constraints

In [20]:
variables_per_passenger

{('0',
  '1'): {'one_one': [('0_1_AZ20271022EBGTPH6882', 'AZ20271024EBGTPH5654'),
   ('0_1_AZ20271022EBGTPH6882', 'AZ20271023EBGTPH5654'),
   ('0_1_AZ20271022EBGTPH6882', 'AZ20271022EBGTPH18974'),
   ('0_1_AZ20271022EBGTPH6882', 'AZ20271023EBGTPH18974'),
   ('0_1_AZ20271022EBGTPH6882', 'AZ20271023EBGTPH1141'),
   ('0_1_AZ20271022EBGTPH6882', 'AZ20271022EBGTPH1141'),
   ('0_1_AZ20271022EBGTPH6882', 'AZ20271023EBGTPH6882'),
   ('0_1_AZ20271022EBGTPH6882', 'AZ20271024EBGTPH6882')], 'one_multi': [], 'multi_one': [('1_1_AZ20271021TPHRHP12465', 'AZ20271023TPHRHP2755'),
   ('1_1_AZ20271021TPHRHP12465', 'AZ20271022TPHRHP12465'),
   ('1_1_AZ20271021TPHRHP12465', 'AZ20271023TPHRHP12465'),
   ('1_1_AZ20271021TPHRHP12465', 'AZ20271022TPHRHP16724'),
   ('1_1_AZ20271021TPHRHP12465', 'AZ20271021TPHRHP16724'),
   ('1_1_AZ20271021TPHRHP12465', 'AZ20271022TPHRHP6701'),
   ('1_1_AZ20271021TPHRHP12465', 'AZ20271023TPHRHP6701'),
   ('1_1_AZ20271021TPHRHP12465', 'AZ20271022TPHRHP5063'),
   ('1_1_AZ20271021T

In [21]:
pnr_list[1].oper_od_orig_cd

'KHL'

In [22]:
pnr_list[1].orig_cd

'TPH'

In [23]:
pnr_list[1].oper_od_dest_cd

'RHP'

In [24]:
pnr_list[1].dest_cd

'RHP'

In [25]:
flights_2legs_out_in_dic[(str(pnr_list[1].orig_cd), str(pnr_list[1].dest_cd))]

[]

In [26]:
available_flights_df = 'data_files/PRMI-DM-AVAILABLE_FLIGHTS.csv'  
available_flights_df = pd.read_csv(available_flights_df)

In [27]:
pnr_df = pd.read_csv("data_files/PRMI_DM_ALL_PNRs.csv")
pnr_df[pnr_df["RECLOC"] == 1]

,RECLOC,CREATION_DTZ,CABIN_CD,COS_CD,OPER_OD_ORIG_CD,OPER_OD_DEST_CD,DEP_KEY,DEP_DT,ORIG_CD,DEST_CD,FLT_NUM,DEP_DTML,ARR_DTML,DEP_DTMZ,ARR_DTMZ,OD_BROKEN_IND,PAX_CNT,CVM,CONN_TIME_MINS
1,1,2027-09-03,Y,21,KHL,RHP,AZ20271021KHLTPH9400,2027-10-21,KHL,TPH,9400,2027-10-21 11:38,2027-10-21 16:29,2027-10-21 14:38,2027-10-21 21:29,0,3,5.68905,NaN
2,1,2027-09-03,Y,20,RHP,KHL,AZ20271104RHPTPH3571,2027-11-04,RHP,TPH,3571,2027-11-04 19:57,2027-11-04 23:25,2027-11-05 0:57,2027-11-05 4:25,0,3,5.68905,NaN
3,1,2027-09-03,Y,20,RHP,KHL,AZ20271104TPHKHL548,2027-11-04,TPH,KHL,548,2027-11-05 0:51,2027-11-05 9:53,2027-11-05 5:51,2027-11-05 12:53,0,3,5.68905,86.0
4,1,2027-09-03,Y,21,KHL,RHP,AZ20271021TPHRHP12465,2027-10-21,TPH,RHP,12465,2027-10-21 18:25,2027-10-21 21:55,2027-10-21 23:25,2027-10-22 2:55,0,3,5.68905,116.0


In [28]:
print("Number of total constraints: ", len(total_constraints))

Number of total constraints:  19


## Now we obtain all of the model

In [29]:
# Cost function
objective = sop.Objective(Total_cost, "maximize")

In [30]:
Reaccommodation = sop.Model(objective, total_constraints)

In [31]:
variables_per_passenger[("0","1")]["multi_multi"]

[]

In [ ]:
flights_2legs_out_in_dic

## We obtain the number of variables

In [ ]:
n_one_one = 0
n_one_multi = 0
n_multi_one = 0
n_multi_multi = 0
n_forced_one = 0


for key in variables_per_passenger:
    n_one_one = n_one_one + len( variables_per_passenger[key]["one_one"] )
    n_one_multi = n_one_multi + len( variables_per_passenger[key]["one_multi"] )
    n_multi_one = n_multi_one + len( variables_per_passenger[key]["multi_one"] )
    n_multi_multi = n_multi_multi + len( variables_per_passenger[key]["multi_multi"] )
    n_forced_one = n_forced_one + len( variables_per_passenger[key]["forced_one"] )
    
n_variables = n_one_one + n_one_multi + n_multi_one + n_multi_multi + n_forced_one

In [ ]:
print("********************************************************")
print("Number of one_one: ", n_one_one)
print("Number of one_multi: ", n_one_multi)
print("Number of multi_one: ", n_multi_one)
print("Number of multi_multi: ", n_multi_multi)
print("Number of forced_one: ", n_forced_one)
print("Total number of variables: ", n_variables)

In [ ]:
# Finally we OPTIMIZE!!!
print("Almost optimizing...")
#result = Reaccommodation.optimize(solver = "simulated_annealing", num_solution = 5, timeout = 60)

# The total number of the search space is
# n_search_space = 2**n_variables 
# because these are binary variables
# Let us optimize working with a fraction of it

n_factor = 2**5 # 1 / 2**n_factor of the search space
n_search_space = (2**n_variables)/(2**n_factor)

time_per_sample = 60*60*1 # 1 Hour (excessively high!)
time_per_sample = 60*5 # 5 minutes

In [ ]:
# This one supports up to 1M variables!

solver = "dwave_hybrid"
#solver = "simulated_annealing"


# Import solver ! 
import neal
#solver=neal.SimulatedAnnealingSampler()

print(r"Optimizing!!! Solver: {}".format(solver))

result = Reaccommodation.optimize(solver=solver, num_solutions = int(n_search_space), 
                                  time_limit=time_per_sample, dwave_api_token = DWAVE_API_TOKEN
                                  )